In [1]:
import os
import pandas as pd
import numpy as np
import s3fs
from sqlalchemy import create_engine
from dotenv import load_dotenv

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

load_dotenv()

False

In [2]:
S3_BUCKET_CURATED = os.getenv('S3_BUCKET_CURATED', 'curated')
MINIO_ACCESS_KEY  = os.getenv('AWS_ACCESS_KEY_ID',     'minioadmin')
MINIO_SECRET_KEY  = os.getenv('AWS_SECRET_ACCESS_KEY', 'minioadmin123')
MINIO_ENDPOINT    = os.getenv('MINIO_ENDPOINT',        'minio:9000')
MINIO_SECURE      = os.getenv('MINIO_SECURE', 'false').lower() == 'true'

POSTGRES_HOST     = os.getenv('POSTGRES_HOST',     'postgres')
POSTGRES_PORT     = os.getenv('POSTGRES_PORT',     '5432')
POSTGRES_DB       = os.getenv('POSTGRES_DB',       'oil_pipeline')
POSTGRES_USER     = os.getenv('POSTGRES_USER',     'oil_user')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'oil_password')

In [3]:
def get_s3fs():
    protocol = 'https' if MINIO_SECURE else 'http'
    return s3fs.S3FileSystem(
        key=MINIO_ACCESS_KEY,
        secret=MINIO_SECRET_KEY,
        client_kwargs={'endpoint_url': f'{protocol}://{MINIO_ENDPOINT}'},
    )

def get_pg_engine():
    return create_engine(
        f'postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}'
        f'@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}'
    )

In [4]:
engine = get_pg_engine()

sensors  = pd.read_sql('SELECT * FROM pump_sensors ORDER BY pump_id, timestamp', engine)
failures = pd.read_sql('SELECT * FROM pump_failures ORDER BY pump_id, failure_date', engine)
pumps    = pd.read_sql('SELECT * FROM pumps', engine)

sensors['timestamp']     = pd.to_datetime(sensors['timestamp'])
failures['failure_date'] = pd.to_datetime(failures['failure_date'])

print(f'pump_sensors:  {sensors.shape}')
print(f'pump_failures: {failures.shape}')
print(f'pumps:         {pumps.shape}')
sensors.head(3)

pump_sensors:  (696, 8)
pump_failures: (3, 5)
pumps:         (5, 6)


,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure
0,1,1,2025-10-01 00:00:00,72.3,2.1,58.2,1470.0,122.4
1,2,1,2025-10-01 03:00:00,72.6,2.0,58.4,1472.0,122.5
2,3,1,2025-10-01 06:00:00,73.1,2.2,58.6,1474.0,122.6


In [5]:
print(failures)
print(pumps)

   failure_id  pump_id        failure_date       failure_type  downtime_hours
0           1        1 2025-10-04 02:00:00        Overheating             6.5
1           2        3 2025-10-04 06:00:00  Bearing vibration             8.0
2           3        5 2025-10-04 08:00:00   Electrical fault            10.2
   pump_id  well_id             type install_date  manufacturer   model
0        1        1          ESP-500   2022-04-10           NOV  ALX500
1        2        1          ESP-500   2022-04-15           NOV  ALX500
2        3        2  Centrifugal-300   2021-12-01        Borets   CF300
3        4        3  Progressive-100   2023-01-12  Schlumberger   PG100
4        5        5          ESP-450   2022-07-21   BakerHughes   BH450


In [6]:
SENSOR_COLS = ['temperature', 'vibration', 'current', 'rpm']
Z_THRESHOLD = 3.0

zscore_df = sensors.copy()

for col in SENSOR_COLS:
    zscore_df[f'z_{col}'] = zscore_df.groupby('pump_id')[col].transform(
        lambda x: (x - x.mean()) / x.std()
    ).abs()

z_cols = [f'z_{c}' for c in SENSOR_COLS]
zscore_df['max_z']        = zscore_df[z_cols].max(axis=1)
zscore_df['is_anomaly_z'] = zscore_df['max_z'] > Z_THRESHOLD

total_z = zscore_df['is_anomaly_z'].sum()
print(f'Всего аномалий (Z-score > {Z_THRESHOLD}): {total_z}')
print(f'Доля аномалий: {total_z / len(zscore_df) * 100:.2f}%')
print()
print('Аномалии по насосам:')
print(zscore_df.groupby('pump_id')['is_anomaly_z'].sum())

Всего аномалий (Z-score > 3.0): 24
Доля аномалий: 3.45%

Аномалии по насосам:
pump_id
1     8
3     6
5    10
Name: is_anomaly_z, dtype: int64


In [7]:
SENSOR_COLS = ['temperature', 'vibration', 'current', 'rpm']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(sensors[SENSOR_COLS])

iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
sensors['anomaly_if']    = iso.fit_predict(X_scaled)
sensors['anomaly_score'] = iso.score_samples(X_scaled)

sensors['is_anomaly_if'] = sensors['anomaly_if'] == -1

total_if = sensors['is_anomaly_if'].sum()
print(f'Всего аномалий (Isolation Forest): {total_if}')
print(f'Доля аномалий: {total_if / len(sensors) * 100:.2f}%')
print()
print('Аномалии по насосам (IF):')
print(sensors.groupby('pump_id')['is_anomaly_if'].sum())

Всего аномалий (Isolation Forest): 35
Доля аномалий: 5.03%

Аномалии по насосам (IF):
pump_id
1     9
3     4
5    22
Name: is_anomaly_if, dtype: int64


In [8]:
sensors['is_anomaly_z'] = zscore_df['is_anomaly_z'].values
sensors['max_z']        = zscore_df['max_z'].values
for col in SENSOR_COLS:
    sensors[f'z_{col}'] = zscore_df[f'z_{col}'].values

sensors['is_anomaly'] = sensors['is_anomaly_z'] | sensors['is_anomaly_if']

print('Итоговые аномалии по насосам (Z-score ИЛИ IF):')
print(sensors.groupby('pump_id')['is_anomaly'].sum())
print()
print(f'Всего уникальных аномальных записей: {sensors["is_anomaly"].sum()}')

Итоговые аномалии по насосам (Z-score ИЛИ IF):
pump_id
1     9
3     6
5    22
Name: is_anomaly, dtype: int64

Всего уникальных аномальных записей: 37


In [9]:
pre_failure_rows = []

for _, failure in failures.iterrows():
    pump_id      = failure['pump_id']
    failure_time = failure['failure_date']
    failure_type = failure['failure_type']
    
    window_start = failure_time - pd.Timedelta(hours=24)
    
    mask = (
        (sensors['pump_id'] == pump_id) &
        (sensors['timestamp'] >= window_start) &
        (sensors['timestamp'] <= failure_time)
    )
    
    pre = sensors[mask].copy()
    pre['failure_date']        = failure_time
    pre['failure_type']        = failure_type
    pre['hours_before_failure'] = (failure_time - pre['timestamp']).dt.total_seconds() / 3600
    
    pre_failure_rows.append(pre)

pre_failure_df = pd.concat(pre_failure_rows, ignore_index=True)

print(f'Записей перед отказами: {pre_failure_df.shape}')
print()
print('По насосам:')
print(pre_failure_df.groupby('pump_id')[['vibration','temperature','current','rpm']].mean().round(2))

Записей перед отказами: (18, 21)

По насосам:
         vibration  temperature  current      rpm
pump_id                                          
1             6.53        81.61    63.10  1501.57
3             6.88        78.33    58.73  1471.17
5            17.36        86.78    64.64  1528.00


In [10]:
latest_time = sensors['timestamp'].max()
window_start = latest_time - pd.Timedelta(hours=24)

recent = sensors[sensors['timestamp'] >= window_start].copy()

global_stats = sensors.groupby('pump_id')[SENSOR_COLS].agg(['mean','std'])

risk_rows = []
for pump_id in sensors['pump_id'].unique():
    pump_recent = recent[recent['pump_id'] == pump_id]
    if len(pump_recent) == 0:
        continue
    
    z_scores = []
    for col in SENSOR_COLS:
        mean = global_stats.loc[pump_id, (col, 'mean')]
        std  = global_stats.loc[pump_id, (col, 'std')]
        if std > 0:
            z = abs((pump_recent[col].mean() - mean) / std)
            z_scores.append(z)
    
    n_anomalies = pump_recent['is_anomaly'].sum()
    
    n_failures = len(failures[failures['pump_id'] == pump_id])
    
    risk_score = round(
        np.mean(z_scores) * 0.5 +
        (n_anomalies / max(len(pump_recent), 1)) * 10 * 0.3 +
        n_failures * 0.2,
        3
    )
    
    risk_rows.append({
        'pump_id':          pump_id,
        'pump_type':        pumps[pumps['pump_id'] == pump_id]['type'].values[0],
        'avg_vibration':    round(pump_recent['vibration'].mean(), 3),
        'avg_temperature':  round(pump_recent['temperature'].mean(), 2),
        'avg_current':      round(pump_recent['current'].mean(), 2),
        'avg_rpm':          round(pump_recent['rpm'].mean(), 1),
        'anomalies_24h':    int(n_anomalies),
        'total_failures':   int(n_failures),
        'risk_score':       risk_score,
    })

risk_df = pd.DataFrame(risk_rows).sort_values('risk_score', ascending=False).reset_index(drop=True)
print('Risk Score по насосам:')
risk_df

Risk Score по насосам:


,pump_id,pump_type,avg_vibration,avg_temperature,avg_current,avg_rpm,anomalies_24h,total_failures,risk_score
0,5,ESP-450,10.694,76.79,56.14,1460.4,7,1,2.992
1,3,Centrifugal-300,1.990,70.09,54.47,1431.8,0,1,0.306
2,1,ESP-500,2.043,73.19,58.49,1472.3,0,1,0.300


In [11]:
mart_anomalies_time = sensors[[
    'pump_id', 'timestamp', 'temperature', 'vibration',
    'current', 'rpm', 'pressure',
    'is_anomaly_z', 'is_anomaly_if', 'is_anomaly',
    'max_z', 'anomaly_score'
]].copy()

mart_anomalies_time['date'] = mart_anomalies_time['timestamp'].dt.date
mart_anomalies_time['hour'] = mart_anomalies_time['timestamp'].dt.hour

print(mart_anomalies_time.shape)
mart_anomalies_time.head(3)

(696, 14)


,pump_id,timestamp,temperature,vibration,current,rpm,pressure,is_anomaly_z,is_anomaly_if,is_anomaly,max_z,anomaly_score,date,hour
0,1,2025-10-01 00:00:00,72.3,2.1,58.2,1470.0,122.4,False,False,False,0.564178,-0.397245,2025-10-01,0
1,1,2025-10-01 03:00:00,72.6,2.0,58.4,1472.0,122.5,False,False,False,0.426252,-0.395530,2025-10-01,3
2,1,2025-10-01 06:00:00,73.1,2.2,58.6,1474.0,122.6,False,False,False,0.183188,-0.384369,2025-10-01,6


In [12]:
mart_pre_failure = pre_failure_df[[
    'pump_id', 'timestamp', 'failure_date', 'failure_type',
    'hours_before_failure', 'vibration', 'temperature', 'current', 'rpm'
]].copy()

print(mart_pre_failure.shape)
mart_pre_failure.head(3)

(18, 9)


,pump_id,timestamp,failure_date,failure_type,hours_before_failure,vibration,temperature,current,rpm
0,1,2025-10-03 03:00:00,2025-10-04 02:00:00,Overheating,23.0,4.5,78.2,61.5,1495.0
1,1,2025-10-03 06:00:00,2025-10-04 02:00:00,Overheating,20.0,5.0,79.1,62.0,1498.0
2,1,2025-10-03 09:00:00,2025-10-04 02:00:00,Overheating,17.0,5.6,80.5,62.5,1500.0


In [13]:
engine = get_pg_engine()

mart_anomalies_time.to_sql('mart_anomalies_time',  engine, if_exists='replace', index=False)
mart_pre_failure.to_sql(   'mart_pre_failure',      engine, if_exists='replace', index=False)
risk_df.to_sql(            'mart_pump_risk_score',  engine, if_exists='replace', index=False)

print('Все марты сохранены в PostgreSQL:')
print('  - mart_anomalies_time')
print('  - mart_pre_failure')
print('  - mart_pump_risk_score')

Все марты сохранены в PostgreSQL:
  - mart_anomalies_time
  - mart_pre_failure
  - mart_pump_risk_score


In [14]:
print(pd.read_sql('SELECT * FROM mart_anomalies_time  LIMIT 3', engine))
print()
print(pd.read_sql('SELECT * FROM mart_pre_failure      LIMIT 3', engine))
print()
print(pd.read_sql('SELECT * FROM mart_pump_risk_score', engine))

   pump_id           timestamp  temperature  vibration  current     rpm  \
0        1 2025-10-01 00:00:00         72.3        2.1     58.2  1470.0   
1        1 2025-10-01 03:00:00         72.6        2.0     58.4  1472.0   
2        1 2025-10-01 06:00:00         73.1        2.2     58.6  1474.0   

   pressure  is_anomaly_z  is_anomaly_if  is_anomaly     max_z  anomaly_score  \
0     122.4         False          False       False  0.564178      -0.397245   
1     122.5         False          False       False  0.426252      -0.395530   
2     122.6         False          False       False  0.183188      -0.384369   

         date  hour  
0  2025-10-01     0  
1  2025-10-01     3  
2  2025-10-01     6  

   pump_id           timestamp        failure_date failure_type  \
0        1 2025-10-03 03:00:00 2025-10-04 02:00:00  Overheating   
1        1 2025-10-03 06:00:00 2025-10-04 02:00:00  Overheating   
2        1 2025-10-03 09:00:00 2025-10-04 02:00:00  Overheating   

   hours_before_

In [15]:
print('=' * 55)
print('Pump Anomaly Detection — Summary')
print('=' * 55)
print(f'Всего записей pump_sensors:       {len(sensors)}')
print(f'Всего отказов pump_failures:       {len(failures)}')
print()
print(f'Аномалии Z-score (порог={Z_THRESHOLD}):   {sensors["is_anomaly_z"].sum()}')
print(f'Аномалии Isolation Forest:         {sensors["is_anomaly_if"].sum()}')
print(f'Итого уникальных аномалий:         {sensors["is_anomaly"].sum()}')
print()
print(f'Записей перед отказами (окно 24ч): {len(pre_failure_df)}')
print()
print('Насосы с наибольшим Risk Score:')
print(risk_df[['pump_id','pump_type','anomalies_24h','total_failures','risk_score']].to_string(index=False))
print('=' * 55)
print()
print('Марты сохранены в PostgreSQL:')
print('  - mart_anomalies_time   → Аномалии по времени')
print('  - mart_pre_failure      → Вибрация перед отказом')
print('  - mart_pump_risk_score  → Risk Score по насосам')

Pump Anomaly Detection — Summary
Всего записей pump_sensors:       696
Всего отказов pump_failures:       3

Аномалии Z-score (порог=3.0):   24
Аномалии Isolation Forest:         35
Итого уникальных аномалий:         37

Записей перед отказами (окно 24ч): 18

Насосы с наибольшим Risk Score:
 pump_id       pump_type  anomalies_24h  total_failures  risk_score
       5         ESP-450              7               1       2.992
       3 Centrifugal-300              0               1       0.306
       1         ESP-500              0               1       0.300

Марты сохранены в PostgreSQL:
  - mart_anomalies_time   → Аномалии по времени
  - mart_pre_failure      → Вибрация перед отказом
  - mart_pump_risk_score  → Risk Score по насосам
